# EstateMind — Anomaly Detection & Market Opportunity Scoring
**`anomaly_detection.ipynb`**

> **DSO3** — Detect significant price discrepancies between observed and expected market values, with explainable indicators for decision support.

| Input | Output |
|-------|--------|
| `tunisia_realestate_cleaned.csv` (same as Price_predict) | `anomaly_report.csv` — scored dataset with flags, confidence, and opportunity labels |
| `estatemind_price_best_model.pkl` (trained in Price_predict) | `anomaly_map.html` — interactive Folium map of anomalies |

**Workflow**: Load data → Reconstruct features → Load price model → Compute price gap → Score anomalies (Isolation Forest + Z-score + IQR) → SHAP explanation → Export report + map

## 1. Setup and Imports

In [1]:
import os, warnings, json
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from tqdm.auto import tqdm

from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.ensemble import IsolationForest
from sklearn.pipeline import make_pipeline
from scipy import stats

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
PALETTE = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D', '#3B1F2B', '#44BBA4']
plt.rcParams.update({'figure.dpi': 130, 'figure.facecolor': 'white'})
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
print('Imports OK.')

c:\EstateMind\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports OK.


## 2. Load Dataset

In [2]:
DATA_PATH = Path('../tunisia_realestate_cleaned.csv')

fallback_paths = [
    Path('tunisia_realestate_cleaned.csv'),
    Path('final_dataset_all_sources_endpoint.csv'),
    Path(r'C:\EstateMind\datasets\Final_dataset.csv'),
    Path(r'C:\EstateMind\datasets\tunisia_realestate_cleaned.csv'),
    Path(r'C:\EstateMind\datasets\final_dataset_all_sources_endpoint.csv'),
]

if not DATA_PATH.exists():
    for p in fallback_paths:
        if p.exists():
            DATA_PATH = p
            print(f'Using fallback: {DATA_PATH}')
            break
    else:
        raise FileNotFoundError(f"CSV not found. Tried: {[str(p) for p in fallback_paths]}")

df = pd.read_csv(DATA_PATH, low_memory=False)
print(f'Dataset: {df.shape[0]:,} rows x {df.shape[1]} cols')

for col in ['titre', 'description', 'prix', 'surface', 'pieces',
            'gouvernerat', 'ville', 'type', 'contrat', 'latitude', 'longitude']:
    if col not in df.columns:
        df[col] = np.nan

Dataset: 204,745 rows x 48 cols


## 3. Feature Engineering
Mirrors the Gold+ pipeline from `Price_predict.ipynb` so features align with the trained model.

In [3]:
# --- Tabular base ---
TABULAR_COLS = [
    'surface', 'pieces', 'etage', 'prix',
    'has_ascenseur', 'has_balcon', 'has_chaffage', 'has_climatisation',
    'has_garage', 'has_gardien', 'has_jardin', 'has_parking', 'has_piscine', 'has_terrasse',
    'gouvernerat', 'ville', 'type', 'contrat', 'latitude', 'longitude',
]
TABULAR_COLS = [c for c in TABULAR_COLS if c in df.columns]

df_gp = df[TABULAR_COLS].copy()
df_gp = df_gp.dropna(subset=['prix']).copy()

if 'surface' in df_gp.columns:
    df_gp['price_per_m2']    = df_gp['prix'] / df_gp['surface'].replace(0, np.nan)
    df_gp['surface_per_piece'] = df_gp['surface'] / df_gp['pieces'].replace(0, np.nan)
if 'has_ascenseur' in df_gp.columns and 'etage' in df_gp.columns:
    df_gp['etage_x_ascenseur'] = df_gp['etage'].fillna(0) * df_gp['has_ascenseur'].fillna(0)

amenity_cols = [c for c in ['has_ascenseur','has_balcon','has_chaffage','has_climatisation',
                             'has_garage','has_gardien','has_jardin','has_parking',
                             'has_piscine','has_terrasse'] if c in df_gp.columns]
df_gp['total_amenities'] = df_gp[amenity_cols].fillna(0).sum(axis=1)

if 'latitude' in df_gp.columns and 'longitude' in df_gp.columns:
    df_gp['latitude']  = df_gp['latitude'].fillna(df_gp['latitude'].median())
    df_gp['longitude'] = df_gp['longitude'].fillna(df_gp['longitude'].median())
    def haversine(lat1, lon1, lat2, lon2):
        R = 6371
        phi1, phi2 = np.radians(lat1), np.radians(lat2)
        a = np.sin(np.radians(lat2-lat1)/2)**2 + np.cos(phi1)*np.cos(phi2)*np.sin(np.radians(lon2-lon1)/2)**2
        return R * 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))
    df_gp['dist_to_center'] = haversine(df_gp['latitude'], df_gp['longitude'], 36.8065, 10.1815)

# Merge NLP/Vision/BGE if available
FEATURE_DIR = Path(r'C:\EstateMind\datasets')
def load_feature_csv(path, label):
    if path.exists():
        feat = pd.read_csv(path, index_col=0)
        print(f'  {label}: {feat.shape[0]:,} rows x {feat.shape[1]} cols')
        return feat
    print(f'  {label}: not found — skipping')
    return pd.DataFrame()

nlp_flat = load_feature_csv(FEATURE_DIR / 'nlp_features.csv',     'NLP')
vis_df   = load_feature_csv(FEATURE_DIR / 'vision_features.csv',  'Vision')
bge_df   = load_feature_csv(FEATURE_DIR / 'bge_pca_features.csv', 'BGE')

if len(nlp_flat) > 0: df_gp = df_gp.join(nlp_flat, how='left')
if len(vis_df)   > 0: df_gp = df_gp.join(vis_df,   how='left')
if len(bge_df)   > 0: df_gp = df_gp.join(bge_df,   how='left')

print(f'Gold+ dataset: {df_gp.shape[0]:,} rows x {df_gp.shape[1]} cols')

  NLP: 100 rows x 30 cols
  Vision: 80,769 rows x 13 cols
  BGE: not found — skipping
Gold+ dataset: 184,374 rows x 68 cols


## 4. Load Price Model & Compute Predicted Prices
Load the trained model from `Price_predict.ipynb` and compute `predicted_price` for all rows.

In [4]:
import joblib
import category_encoders as ce

MODEL_PATH = Path('estatemind_price_best_model.pkl')
if not MODEL_PATH.exists():
    MODEL_PATH = Path(r'C:\EstateMind\estatemind_price_best_model.pkl')

if not MODEL_PATH.exists():
    raise FileNotFoundError(f"Model not found at {MODEL_PATH}. Run Price_predict.ipynb first.")

loaded_model = joblib.load(MODEL_PATH)
print(f'Model loaded: {MODEL_PATH}')

Model loaded: estatemind_price_best_model.pkl


In [ ]:
# Reconstruct features using same logic as Price_predict (no meta file needed)
TARGET = "prix"

NUM_FEATS = [
    c for c in df_gp.columns
    if df_gp[c].dtype in [np.float64, np.float32, np.int64, np.int32, float, int]
    and c != TARGET
    and df_gp[c].notna().mean() > 0.05
]
CAT_FEATS = [c for c in ['type', 'contrat', 'gouvernerat'] if c in df_gp.columns]

print(f"Features — {len(NUM_FEATS)} num + {len(CAT_FEATS)} cat")

df_ml = df_gp[[TARGET] + NUM_FEATS + CAT_FEATS].copy()
df_ml[TARGET] = pd.to_numeric(df_ml[TARGET], errors="coerce")
df_ml = df_ml.dropna(subset=[TARGET])

X_all = df_ml[NUM_FEATS + CAT_FEATS].copy()
y_all = df_ml[TARGET].astype(float)

te    = ce.TargetEncoder(cols=CAT_FEATS, smoothing=10)
X_enc = te.fit_transform(X_all, y_all)
med   = X_enc.median()
X_enc = X_enc.fillna(med)

# Align to model's expected features
model_feature_count = loaded_model.n_features_in_
if X_enc.shape[1] != model_feature_count:
    X_enc = X_enc.iloc[:, :model_feature_count]
    print(f"Trimmed to {model_feature_count} features to match model.")

print(f"X_enc shape: {X_enc.shape}")

y_pred = loaded_model.predict(X_enc)
print(f"Predictions computed for {len(y_pred):,} properties.")

Features — 32 num + 3 cat


## 5. Price Gap Computation
Compute the gap between observed and predicted price — the core anomaly signal.

In [ ]:
df_scored = df_ml.copy()
df_scored['predicted_price'] = y_pred
df_scored['price_gap']       = df_scored[TARGET] - df_scored['predicted_price']
df_scored['price_gap_pct']   = (df_scored['price_gap'] / df_scored[TARGET]) * 100

# Keep original text columns for reporting
for col in ['titre', 'description', 'adresse']:
    if col in df.columns:
        df_scored[col] = df.loc[df_scored.index, col] if col in df.columns else np.nan

print(df_scored[['prix','predicted_price','price_gap','price_gap_pct']].describe().round(2))

In [ ]:
# Distribution of price gap
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Price Gap Distribution', fontsize=14, fontweight='bold')

clip = df_scored['price_gap_pct'].clip(-200, 200)
axes[0].hist(df_scored['price_gap'] / 1000, bins=60, color=PALETTE[0], edgecolor='white', alpha=0.85)
axes[0].axvline(0, color='red', lw=2, linestyle='--', label='Zero gap')
axes[0].set_xlabel('Price Gap (k TND)'); axes[0].set_title('Absolute Gap (Actual − Predicted)')
axes[0].legend()

axes[1].hist(clip, bins=60, color=PALETTE[1], edgecolor='white', alpha=0.85)
axes[1].axvline(0, color='red', lw=2, linestyle='--')
axes[1].axvline(-25, color='orange', lw=1.5, linestyle=':', label='±25% threshold')
axes[1].axvline(25,  color='orange', lw=1.5, linestyle=':')
axes[1].set_xlabel('Price Gap (%)'); axes[1].set_title('Relative Gap % (clipped ±200%)')
axes[1].legend()

plt.tight_layout()
plt.savefig('price_gap_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Multi-Method Anomaly Scoring

Three complementary methods are combined into a single **anomaly score**:

| Method | What it catches |
|--------|----------------|
| **Z-score** | Listings far from the mean gap in standard deviations |
| **IQR fence** | Listings beyond the 1.5×IQR whiskers |
| **Isolation Forest** | Multivariate anomalies across all features |

A listing is flagged when **≥ 2 out of 3** methods agree.

In [ ]:
# ── Method 1: Z-score on price_gap_pct ──
z_scores = np.abs(stats.zscore(df_scored['price_gap_pct'].fillna(0)))
df_scored['zscore_anomaly'] = (z_scores > 2.5).astype(int)

# ── Method 2: IQR fence ──
Q1  = df_scored['price_gap_pct'].quantile(0.25)
Q3  = df_scored['price_gap_pct'].quantile(0.75)
IQR = Q3 - Q1
lower_fence = Q1 - 2.0 * IQR
upper_fence = Q3 + 2.0 * IQR
df_scored['iqr_anomaly'] = (
    (df_scored['price_gap_pct'] < lower_fence) |
    (df_scored['price_gap_pct'] > upper_fence)
).astype(int)

# ── Method 3: Isolation Forest (multivariate) ──
iso_features = ['prix', 'predicted_price', 'price_gap_pct'] + [
    c for c in ['surface', 'pieces', 'dist_to_center', 'total_amenities', 'price_per_m2']
    if c in df_scored.columns
]
X_iso = df_scored[iso_features].fillna(df_scored[iso_features].median())

iso = IsolationForest(
    n_estimators=200,
    contamination=0.05,   # expect ~5% anomalies
    random_state=RANDOM_SEED,
    n_jobs=-1
)
df_scored['iso_score']   = iso.fit_predict(X_iso)   # -1 = anomaly
df_scored['iso_anomaly'] = (df_scored['iso_score'] == -1).astype(int)
df_scored['iso_raw_score'] = iso.decision_function(X_iso)  # lower = more anomalous

# ── Consensus flag: ≥2 methods agree ──
df_scored['anomaly_votes'] = (
    df_scored['zscore_anomaly'] +
    df_scored['iqr_anomaly'] +
    df_scored['iso_anomaly']
)
df_scored['is_anomaly'] = (df_scored['anomaly_votes'] >= 2).astype(int)

n_anomalies = df_scored['is_anomaly'].sum()
print(f"Anomalies detected (≥2/3 methods): {n_anomalies:,} / {len(df_scored):,} ({n_anomalies/len(df_scored)*100:.1f}%)")
print(f"  Z-score only  : {df_scored['zscore_anomaly'].sum():,}")
print(f"  IQR only      : {df_scored['iqr_anomaly'].sum():,}")
print(f"  Isolation F.  : {df_scored['iso_anomaly'].sum():,}")

## 7. Opportunity & Risk Labelling
Each anomaly is classified into one of four categories based on the direction and magnitude of the price gap.

In [ ]:
def label_opportunity(row):
    if row['is_anomaly'] == 0:
        return 'Normal'
    gap = row['price_gap_pct']
    if gap < -30:
        return 'Strong Opportunity'   # actual << predicted → underpriced
    elif gap < -10:
        return 'Opportunity'
    elif gap > 30:
        return 'Overpriced Risk'       # actual >> predicted → overpriced
    elif gap > 10:
        return 'Slight Overpricing'
    else:
        return 'Borderline'

df_scored['opportunity_label'] = df_scored.apply(label_opportunity, axis=1)

label_counts = df_scored['opportunity_label'].value_counts()
print(label_counts.to_string())

In [ ]:
# ── Confidence score ──
# Higher confidence when: iso_raw_score is very negative AND z-score is high
from sklearn.preprocessing import MinMaxScaler

conf_features = pd.DataFrame({
    'abs_gap_pct' : df_scored['price_gap_pct'].abs(),
    'z_score'     : np.abs(stats.zscore(df_scored['price_gap_pct'].fillna(0))),
    'iso_neg'     : -df_scored['iso_raw_score'],   # more negative = more anomalous
    'vote_count'  : df_scored['anomaly_votes'],
})
scaler_conf = MinMaxScaler()
conf_norm = scaler_conf.fit_transform(conf_features.fillna(0))
df_scored['anomaly_confidence'] = np.round(conf_norm.mean(axis=1), 3)

# Only meaningful for flagged anomalies
df_scored.loc[df_scored['is_anomaly'] == 0, 'anomaly_confidence'] = 0.0

print(df_scored.loc[df_scored['is_anomaly']==1, 'anomaly_confidence'].describe().round(3))

## 8. Visualisations

In [ ]:
# ── 8.1 Opportunity label distribution ──
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('EstateMind — Anomaly & Opportunity Overview', fontsize=14, fontweight='bold')

color_map = {
    'Normal':             '#95a5a6',
    'Strong Opportunity': '#27ae60',
    'Opportunity':        '#2ecc71',
    'Slight Overpricing': '#e67e22',
    'Overpriced Risk':    '#c0392b',
    'Borderline':         '#8e44ad',
}
ordered = ['Strong Opportunity','Opportunity','Normal','Borderline','Slight Overpricing','Overpriced Risk']
counts  = [label_counts.get(l, 0) for l in ordered]
colors  = [color_map[l] for l in ordered]

bars = axes[0].barh(ordered, counts, color=colors, edgecolor='white')
for bar, val in zip(bars, counts):
    axes[0].text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2,
                 f'{val:,}', va='center', fontsize=9)
axes[0].set_xlabel('Number of listings')
axes[0].set_title('Listings by Opportunity Label')

# ── 8.2 Scatter: actual vs predicted, coloured by label ──
anomalies_only = df_scored[df_scored['is_anomaly'] == 1]
normal_only    = df_scored[df_scored['is_anomaly'] == 0]
clip_q = float(df_scored['prix'].quantile(0.97))

axes[1].scatter(
    normal_only['prix'].clip(upper=clip_q),
    normal_only['predicted_price'].clip(upper=clip_q),
    alpha=0.15, s=5, color='#bdc3c7', label='Normal'
)
for label, grp in anomalies_only.groupby('opportunity_label'):
    axes[1].scatter(
        grp['prix'].clip(upper=clip_q),
        grp['predicted_price'].clip(upper=clip_q),
        alpha=0.6, s=15, color=color_map.get(label, '#e74c3c'), label=label
    )
lo, hi = 0, clip_q
axes[1].plot([lo, hi], [lo, hi], 'k--', lw=1.5, label='Perfect prediction')
axes[1].set_xlabel('Actual Price (TND)')
axes[1].set_ylabel('Predicted Price (TND)')
axes[1].set_title('Actual vs Predicted — Anomalies Highlighted')
axes[1].legend(fontsize=8, markerscale=2)

plt.tight_layout()
plt.savefig('anomaly_overview.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 8.3 Anomaly rate by gouvernorat ──
gov_stats = (
    df_scored.groupby('gouvernerat')
    .agg(total=('is_anomaly','count'), anomalies=('is_anomaly','sum'))
    .assign(anomaly_rate=lambda x: x['anomalies'] / x['total'] * 100)
    .sort_values('anomaly_rate', ascending=False)
    .head(20)
)

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.bar(gov_stats.index, gov_stats['anomaly_rate'],
              color=PALETTE[0], edgecolor='white')
ax.axhline(gov_stats['anomaly_rate'].mean(), color='red', lw=1.5,
           linestyle='--', label=f"Average: {gov_stats['anomaly_rate'].mean():.1f}%")
ax.set_title('Anomaly Rate by Gouvernorat (Top 20)', fontweight='bold')
ax.set_xlabel('Gouvernorat')
ax.set_ylabel('Anomaly Rate (%)')
ax.set_xticklabels(gov_stats.index, rotation=45, ha='right', fontsize=9)
ax.legend()
plt.tight_layout()
plt.savefig('anomaly_rate_by_region.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 8.4 Price gap heatmap: gouvernorat × property type ──
if 'type' in df_scored.columns and 'gouvernerat' in df_scored.columns:
    pivot = (
        df_scored[df_scored['is_anomaly'] == 1]
        .groupby(['gouvernerat', 'type'])['price_gap_pct']
        .median()
        .unstack(fill_value=0)
    )
    top_govs  = df_scored['gouvernerat'].value_counts().head(12).index
    top_types = df_scored['type'].value_counts().head(6).index
    pivot = pivot.loc[pivot.index.isin(top_govs), pivot.columns.isin(top_types)]

    fig, ax = plt.subplots(figsize=(12, 7))
    sns.heatmap(
        pivot, annot=True, fmt='.0f', cmap='RdYlGn_r',
        center=0, linewidths=0.5, ax=ax,
        cbar_kws={'label': 'Median Gap % (negative = underpriced)'}
    )
    ax.set_title('Median Price Gap % by Gouvernorat × Property Type (Anomalies Only)',
                 fontweight='bold')
    plt.tight_layout()
    plt.savefig('anomaly_heatmap.png', dpi=150, bbox_inches='tight')
    plt.show()

## 9. SHAP Explanation — Why Is This Property an Anomaly?
Use SHAP to explain which features pushed the predicted price up or down for the most extreme anomalies.

In [ ]:
try:
    import shap
    print(f'SHAP {shap.__version__}')
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'shap', '-q'])
    import shap
    print('SHAP installed.')

In [ ]:
# Explainer on the loaded model (LightGBM or RF)
try:
    explainer = shap.TreeExplainer(loaded_model)
    SHAP_OK = True
    print('TreeExplainer ready.')
except Exception as e:
    print(f'TreeExplainer failed: {e} — using KernelExplainer as fallback')
    bg = shap.sample(X_enc, 100, random_state=RANDOM_SEED)
    explainer = shap.KernelExplainer(loaded_model.predict, bg)
    SHAP_OK = True

In [ ]:
if SHAP_OK:
    # Compute SHAP on anomalies only (faster)
    X_anomaly = X_enc.loc[df_scored[df_scored['is_anomaly'] == 1].index]
    shap_values = explainer(X_anomaly)

    # Global importance for anomalous listings
    fig, ax = plt.subplots(figsize=(10, 6))
    shap.plots.bar(shap_values, max_display=15, show=False, ax=ax)
    ax.set_title('SHAP Feature Importance — Anomalous Listings', fontweight='bold')
    plt.tight_layout()
    plt.savefig('shap_anomaly_importance.png', dpi=150, bbox_inches='tight')
    plt.show()

    # Beeswarm summary
    plt.figure(figsize=(10, 6))
    shap.plots.beeswarm(shap_values, max_display=15, show=False)
    plt.title('SHAP Beeswarm — Anomalous Listings')
    plt.tight_layout()
    plt.savefig('shap_anomaly_beeswarm.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
if SHAP_OK:
    # Waterfall for the single most anomalous listing
    most_anomalous_idx = df_scored.loc[
        df_scored['is_anomaly'] == 1, 'anomaly_confidence'
    ].idxmax()
    row_pos = list(X_anomaly.index).index(most_anomalous_idx)

    plt.figure(figsize=(10, 5))
    shap.plots.waterfall(shap_values[row_pos], max_display=12, show=False)
    plt.title(f'SHAP Waterfall — Most Anomalous Listing (idx {most_anomalous_idx})', fontweight='bold')
    plt.tight_layout()
    plt.savefig('shap_waterfall_top_anomaly.png', dpi=150, bbox_inches='tight')
    plt.show()

    row = df_scored.loc[most_anomalous_idx]
    print(f"\nTop anomaly details:")
    print(f"  Actual price     : {row['prix']:,.0f} TND")
    print(f"  Predicted price  : {row['predicted_price']:,.0f} TND")
    print(f"  Gap              : {row['price_gap']:+,.0f} TND ({row['price_gap_pct']:+.1f}%)")
    print(f"  Label            : {row['opportunity_label']}")
    print(f"  Confidence       : {row['anomaly_confidence']:.3f}")

## 10. Interactive Map of Anomalies
Plot all anomalous listings on a Folium map, coloured by opportunity label.

In [ ]:
try:
    import folium
    from folium.plugins import MarkerCluster, HeatMap, MiniMap
    FOLIUM_OK = True
    print(f'Folium ready.')
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'folium', '-q'])
    import folium
    from folium.plugins import MarkerCluster, HeatMap, MiniMap
    FOLIUM_OK = True

In [ ]:
if FOLIUM_OK:
    GOV_COORDS = {
        'Tunis':(36.8189,10.1658),'Ariana':(36.8665,10.1647),'Ben Arous':(36.7533,10.2282),
        'Manouba':(36.8093,10.0963),'Nabeul':(36.4561,10.7376),'Zaghouan':(36.4029,10.1434),
        'Bizerte':(37.2744,9.8739),'Beja':(36.7333,9.1833),'Jendouba':(36.5011,8.7757),
        'Le Kef':(36.1825,8.7147),'Siliana':(36.0851,9.3709),'Sousse':(35.8288,10.6363),
        'Monastir':(35.7643,10.8113),'Mahdia':(35.5047,11.0622),'Sfax':(34.7400,10.7600),
        'Kairouan':(35.6781,10.0968),'Kasserine':(35.1672,8.8306),'Sidi Bouzid':(35.0382,9.4849),
        'Gabes':(33.8828,10.0982),'Medenine':(33.3549,10.5055),'Tataouine':(32.9293,10.4509),
        'Gafsa':(34.4250,8.7842),'Tozeur':(33.9197,8.1335),'Kebili':(33.7050,8.9690),
    }
    LABEL_COLORS = {
        'Strong Opportunity': 'green',
        'Opportunity':        'lightgreen',
        'Borderline':         'purple',
        'Slight Overpricing': 'orange',
        'Overpriced Risk':    'red',
        'Normal':             'gray',
    }

    df_map = df_scored[df_scored['is_anomaly'] == 1].copy()
    df_map['_lat'] = pd.to_numeric(df_map.get('latitude', np.nan), errors='coerce')
    df_map['_lon'] = pd.to_numeric(df_map.get('longitude', np.nan), errors='coerce')

    if 'gouvernerat' in df_map.columns:
        for gov, (lat_g, lon_g) in GOV_COORDS.items():
            mask = df_map['_lat'].isna() & df_map['gouvernerat'].str.contains(gov, case=False, na=False)
            df_map.loc[mask, '_lat'] = lat_g + np.random.uniform(-0.05, 0.05, mask.sum())
            df_map.loc[mask, '_lon'] = lon_g + np.random.uniform(-0.05, 0.05, mask.sum())

    df_map = df_map.dropna(subset=['_lat','_lon']).head(2000)

    m = folium.Map(location=[33.8869, 9.5375], zoom_start=7, tiles='CartoDB positron')
    MiniMap(toggle_display=True).add_to(m)

    # Cluster layer
    cluster = MarkerCluster(name='Anomalies (clustered)').add_to(m)

    for _, row in df_map.iterrows():
        color  = LABEL_COLORS.get(row['opportunity_label'], 'gray')
        gap_pct = row['price_gap_pct']
        conf    = row['anomaly_confidence']
        titre   = str(row.get('titre', 'N/A'))[:80]
        popup_html = (
            f"<b>{titre}</b><br>"
            f"Actual: <b>{row['prix']:,.0f} TND</b><br>"
            f"Predicted: {row['predicted_price']:,.0f} TND<br>"
            f"Gap: <b style='color:{'green' if gap_pct<0 else 'red'}'>"
            f"{gap_pct:+.1f}%</b><br>"
            f"Label: <b>{row['opportunity_label']}</b><br>"
            f"Confidence: {conf:.2f}"
        )
        folium.CircleMarker(
            location=[row['_lat'], row['_lon']],
            radius=max(4, min(12, abs(gap_pct) / 10)),
            color=color, fill=True, fill_color=color, fill_opacity=0.7,
            popup=folium.Popup(popup_html, max_width=280),
            tooltip=f"{row['opportunity_label']} | {gap_pct:+.0f}%"
        ).add_to(cluster)

    # Heatmap layer of anomaly confidence
    heat_data = [[r['_lat'], r['_lon'], r['anomaly_confidence']] for _, r in df_map.iterrows()]
    HeatMap(heat_data, name='Anomaly Density', radius=15, blur=10, min_opacity=0.3).add_to(m)

    folium.LayerControl().add_to(m)

    m.save('anomaly_map.html')
    print(f"Map saved: anomaly_map.html ({len(df_map):,} anomalies plotted)")
    m

## 11. Export Anomaly Report

In [ ]:
REPORT_COLS = [
    'prix', 'predicted_price', 'price_gap', 'price_gap_pct',
    'anomaly_votes', 'is_anomaly', 'opportunity_label', 'anomaly_confidence',
    'zscore_anomaly', 'iqr_anomaly', 'iso_anomaly', 'iso_raw_score',
] + (['gouvernerat'] if 'gouvernerat' in df_scored.columns else [])   + (['ville']       if 'ville'       in df_scored.columns else [])   + (['type']        if 'type'        in df_scored.columns else [])   + (['surface']     if 'surface'     in df_scored.columns else [])   + (['pieces']      if 'pieces'      in df_scored.columns else [])   + (['titre']       if 'titre'       in df_scored.columns else [])

report = df_scored[[c for c in REPORT_COLS if c in df_scored.columns]].copy()
report = report.sort_values('anomaly_confidence', ascending=False)

report.to_csv('anomaly_report.csv', index=True, encoding='utf-8-sig')
print(f"anomaly_report.csv saved — {len(report):,} rows")

# Top 10 opportunities
print("\n🟢 Top 10 Investment Opportunities (most underpriced anomalies):")
opps = report[report['opportunity_label'].isin(['Strong Opportunity','Opportunity'])] \
       .head(10)[['prix','predicted_price','price_gap_pct','anomaly_confidence','gouvernerat','type','titre']]
print(opps.to_string())

print("\n🔴 Top 10 Overpriced Risks:")
risks = report[report['opportunity_label'].isin(['Overpriced Risk','Slight Overpricing'])] \
        .head(10)[['prix','predicted_price','price_gap_pct','anomaly_confidence','gouvernerat','type','titre']]
print(risks.to_string())

## 12. Summary

| Output | Description |
|--------|-------------|
| `anomaly_report.csv` | Full scored dataset with flags, labels and confidence |
| `anomaly_map.html` | Interactive Folium map — click any marker for details |
| `anomaly_overview.png` | Actual vs predicted scatter + label counts |
| `anomaly_rate_by_region.png` | Anomaly rate per gouvernorat |
| `anomaly_heatmap.png` | Median gap heatmap by region × property type |
| `shap_anomaly_importance.png` | Global SHAP importance for anomalous listings |
| `shap_waterfall_top_anomaly.png` | Waterfall explanation for the top anomaly |

### How to interpret labels

| Label | Meaning | Action |
|-------|---------|--------|
| 🟢 **Strong Opportunity** | Actual price ≥ 30% below predicted | Investigate — likely underpriced |
| 🟢 **Opportunity** | 10–30% below predicted | Worth monitoring |
| ⚪ **Normal** | Within expected range | No action needed |
| 🟠 **Slight Overpricing** | 10–30% above predicted | Negotiate or avoid |
| 🔴 **Overpriced Risk** | ≥ 30% above predicted | High financial risk |